# ENGS108 Name That LLM — H5 submission (inference only)

Generated by `submission/h5/build_notebook.py`. No training. Qwen3-8B-Base 4-bit NF4 + the exp03 Arm H epoch-6 adapter; one shared forward pass per batch gives the head logits and the pooled hidden state. H5 = equal-weight log-linear blend of the C=0.03 probe (`probe_armh.npz`), the head, and the stylometric model (`stylo_exp03_model.joblib`). `submission.parquet` (`id`, `model`) is written and validated right after test inference; diagnostics come after it and cannot fail the run. The 414-row dev reproduction against the reported H5 (bal-acc 0.4324) is optional (`run_dev_check` in the config, off by default; train.csv is then never read). Internet off.

In [ ]:
import glob, subprocess, sys
wheels = sorted(glob.glob("/kaggle/input/**/bitsandbytes-*.whl", recursive=True))
assert wheels, "attach the engs108-offline-wheels dataset"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", wheels[-1]], check=True)


In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one T4 (as the Arm H kernel)

import glob, hashlib, json, sys, time
import importlib.metadata as metadata

import numpy as np
import pandas as pd
import torch

RUN_START = time.time()
CONFIG = json.loads(r"""{
 "submission": "H5: equal-weight log-linear blend of probe_armh + armh_head + stylo_exp03 (exp03 pre-registered report H5)",
 "inference_only": true,
 "run_dev_check": false,
 "seed": 42,
 "class_names": ["claude-fable-5", "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5",
                 "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash", "glm-5.3",
                 "glm-5.3-flash", "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra",
                 "kimi-k3", "qwen3.8-max"],
 "model": {
  "base_path": "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-base/1",
  "class": "Qwen3ForSequenceClassification via AutoModelForSequenceClassification(num_labels=18, id2label = alphabetical)",
  "quantization": {"load_in_4bit": true, "bnb_4bit_quant_type": "nf4", "bnb_4bit_compute_dtype": "float16",
                   "bnb_4bit_use_double_quant": true},
  "adapter": "engs108-exp03-armh-artifacts/adapter (Arm H epoch 6 = checkpoint-858, modules_to_save=[score])",
  "max_length": 512,
  "batch_size": 8,
  "autocast": "float16"
 },
 "forward": "one shared pass per batch: exp03 armh_core.pooled_forward (hook on the score input, last non-pad token) returns the head logits and the pooled hidden state of the same pass; batches of 8 in file order, which is the order the Arm H kernel used for pooled_hidden_armh.npy (the probe's training/dev features)",
 "blend": {"id": "H5", "components": ["probe_armh", "armh_head", "stylo_exp03"], "weights": [1.0, 1.0, 1.0],
           "method": "exp03/stylo/protocol.py logprob_blend: softmax(sum_k log clip(p_k, 1e-12, 1))",
           "not": "H6 dev-LOPO weights are NOT used"},
 "dev_check": {
  "rows": 414,
  "prompts": 23,
  "expected_balanced_accuracy": 0.4323671497584541,
  "tolerance": 0.001,
  "require_argmax_agreement_vs_blend_H5_dev": 1.0,
  "max_abs_proba_diff_vs_shared_pass_reference": 0.01
 },
 "sha256": {
  "train.csv": "40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707",
  "adapter/adapter_model.safetensors": "dae9601c1f5cfba01da3c1c189c893a7e011ed274e26c50a775fe51a42084354",
  "adapter/adapter_config.json": "b774b665076a1ba2cb0aa34e299815cbf058cf17ac64ab56139932617e0a8a3a",
  "adapter/label_map.json": "820b6c166aa933d1c92902dbcf4f380b188b0de1a4d6e9e228a7e30ac307aff8",
  "probe_armh.npz": "12dc23b8218447a9148407583b1c5fe015e23023a04e387ad66257c1b3604093",
  "stylo_exp03_model.joblib": "4377983503ab5b4afb6675f273ca21f1699c1b9f46bad7c311212e25d1ee87df",
  "blend_H5_dev.csv": "2dcc47082c9089897fd220ebadc52ec2960ae2d639a9e1c0ea78c2ef224fba73",
  "h5_dev_reference.csv": "bb1a2640f5ee2df0b5b28e33b26c095abc72395e7fb549d4b5a563cef1bf7f27"
 },
 "kaggle": {
  "kernel": "ritisingh123/rs2027-engs108-submit-h5",
  "competition": "name-that-llm",
  "datasets": ["ritisingh123/engs108-offline-wheels", "ritisingh123/engs108-exp03-armh-artifacts",
               "ritisingh123/engs108-submission-h5-artifacts"],
  "artifacts_dataset": "ritisingh123/engs108-submission-h5-artifacts",
  "model_sources": ["qwen-lm/qwen-3/Transformers/8b-base/1"],
  "docker_image": "gcr.io/kaggle-private-byod/python@sha256:de153e6d303881930c51286216393d56f56b26458d0661350305a1e8144eea01",
  "machine_shape": "NvidiaTeslaT4",
  "internet": false,
  "gpu": true,
  "leaderboard_submission": false
 }
}
""")
MC, DC, SHA = CONFIG["model"], CONFIG["dev_check"], CONFIG["sha256"]
SEED, MAX_LENGTH, BATCH, N_CLASSES = CONFIG["seed"], MC["max_length"], MC["batch_size"], len(CONFIG["class_names"])
# Optional 414-row dev reproduction (diagnostic). Off for the official run: then train.csv is never hashed or read.
RUN_DEV_CHECK = CONFIG.get("run_dev_check", False)
DATA_DIR = "/kaggle/input/competitions/name-that-llm"
MODEL_PATH = MC["base_path"]
OUT = "/kaggle/working"

versions = {p: metadata.version(p) for p in
            ["torch", "transformers", "peft", "accelerate", "bitsandbytes", "scikit-learn", "numpy", "pandas", "scipy"]}
print(versions)
print("CUDA:", torch.cuda.is_available(), "| visible GPUs:", torch.cuda.device_count(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "GPU required"


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def find_one(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    assert len(hits) == 1, (pattern, hits)
    return hits[0]


ADAPTER_DIR = os.path.dirname(find_one("/kaggle/input/**/engs108-exp03-armh-artifacts/adapter/adapter_config.json"))
H5_DIR = os.path.dirname(find_one("/kaggle/input/**/engs108-submission-h5-artifacts/probe_armh.npz"))
print("adapter:", ADAPTER_DIR, sorted(os.listdir(ADAPTER_DIR)))
print("h5 artifacts:", H5_DIR, sorted(os.listdir(H5_DIR)))


In [ ]:
FEATURES_SRC = r'''"""exp03 stylometric feature groups.

Groups (names used in configs and ablations):
  char    char_wb 3-5 TF-IDF (exp02)
  word    word 1-2 TF-IDF (exp02)
  style   27 exp02 style features (copied verbatim from exp02; test_features checks equality)
  fw      relative frequencies of a fixed English function-word list (dense)
  fwng    function-word n-grams: content words masked by their shape, TF-IDF 1-3 grams
  typo    typography / formatting rates (dense)
  sent    sentence / paragraph structure statistics and word-length histogram (dense)
  shape   token-shape n-grams (Xx, x, X, d, punctuation literal, newline), TF-IDF 1-3 grams
Dense groups are standardised; TF-IDF groups are used as is. Every learned step is a sklearn
transformer, so it is refit inside each CV fold.
"""
import re
import string

import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

# ---------------------------------------------------------------- exp02 style features (verbatim)
PUNCT = set(string.punctuation) | set("—–‘’“”…«»·•")
STYLE_NAMES = ["log_chars", "log_words", "mean_word_len", "n_sentences", "mean_sentence_words", "type_token_ratio",
               "punct_rate", "upper_rate", "digit_rate", "whitespace_rate", "newline_rate",
               "comma_rate", "semicolon_rate", "colon_rate", "em_dash_rate", "en_dash_rate", "hyphen_rate",
               "curly_apos_rate", "straight_apos_rate", "curly_quote_rate", "straight_quote_rate",
               "paren_rate", "exclaim_rate", "question_rate", "ellipsis_rate", "asterisk_rate", "hash_rate"]


def style_features(texts):
    rows = []
    for t in texts:
        n = max(len(t), 1)
        words = re.findall(r"\w+", t)
        sents = [s for s in re.split(r"[.!?]+\s", t) if s.strip()]
        rows.append([
            np.log1p(len(t)), np.log1p(len(words)), np.mean([len(w) for w in words]) if words else 0.0,
            len(sents), len(words) / max(len(sents), 1), len({w.lower() for w in words}) / max(len(words), 1),
            sum(c in PUNCT for c in t) / n, sum(c.isupper() for c in t) / n, sum(c.isdigit() for c in t) / n,
            sum(c.isspace() for c in t) / n, t.count("\n") / n,
            t.count(",") / n, t.count(";") / n, t.count(":") / n, t.count("—") / n, t.count("–") / n, t.count("-") / n,
            t.count("’") / n, t.count("'") / n, (t.count("“") + t.count("”")) / n, t.count('"') / n,
            (t.count("(") + t.count(")")) / n, t.count("!") / n, t.count("?") / n,
            (t.count("…") + t.count("...")) / n, t.count("*") / n, t.count("#") / n,
        ])
    return np.asarray(rows, dtype=np.float64)


# ---------------------------------------------------------------- function words
# Closed-class English words: determiners, pronouns, prepositions, conjunctions, auxiliaries,
# common adverbs/particles and discourse markers. Fixed a priori (not learned from the data).
FUNCTION_WORDS = sorted(set("""
a an the this that these those some any each every either neither no another such what which whose
i me my mine myself we us our ours ourselves you your yours yourself yourselves he him his himself she her hers
herself it its itself they them their theirs themselves one ones oneself who whom whoever whatever whichever
about above across after against along amid among around as at before behind below beneath beside besides
between beyond but by despite down during except for from in inside into like near of off on onto out outside
over past per since than through throughout till to toward towards under underneath unlike until up upon via
with within without
and or nor so yet both whether because although though while whereas if unless once whenever wherever
however therefore thus hence moreover furthermore nevertheless nonetheless meanwhile instead otherwise indeed
also still even just only already again ever never always often sometimes rather quite very too much more most
less least enough almost perhaps maybe merely simply really actually particularly especially
be am is are was were been being have has had having do does did doing will would shall should can could
may might must ought need dare
not here there where when why how then now well all both few many several most other others same own
""".split()))
_WORD_RE = re.compile(r"[A-Za-z]+(?:['’][A-Za-z]+)?")
_FW_INDEX = {w: i for i, w in enumerate(FUNCTION_WORDS)}


def _words(t):
    return _WORD_RE.findall(t)


def function_word_features(texts):
    """Relative frequency (per word) of each function word; contractions split on the apostrophe."""
    out = np.zeros((len(texts), len(FUNCTION_WORDS)))
    for r, t in enumerate(texts):
        ws = [w.lower().replace("’", "'").split("'")[0] for w in _words(t)]
        for w in ws:
            j = _FW_INDEX.get(w)
            if j is not None:
                out[r, j] += 1
        out[r] /= max(len(ws), 1)
    return out


def _word_shape(w):
    if w.isdigit():
        return "d"
    if w.isupper() and len(w) > 1:
        return "X"
    if w[0].isupper():
        return "Xx"
    return "x"


_TOKEN_RE = re.compile(r"\n|[A-Za-z]+(?:['’][A-Za-z]+)?|\d+(?:[.,]\d+)*|[^\w\s]")


def _tokens(t):
    return _TOKEN_RE.findall(t)


def fw_masked_docs(texts):
    """Keep function words (lower-cased) and punctuation; replace other tokens by their shape."""
    docs = []
    for t in texts:
        out = []
        for tok in _tokens(t):
            if tok == "\n":
                out.append("NL")
            elif tok[0].isalpha():
                base = tok.lower().replace("’", "'").split("'")[0]
                out.append(base if base in _FW_INDEX else "<" + _word_shape(tok) + ">")
            elif tok[0].isdigit():
                out.append("<d>")
            else:
                out.append(tok)
        docs.append(" ".join(out))
    return docs


def shape_docs(texts):
    """Token-shape sequence: word shape + length bucket (s<=3, m<=6, l), digits 'd', punctuation literal."""
    docs = []
    for t in texts:
        out = []
        for tok in _tokens(t):
            if tok == "\n":
                out.append("NL")
            elif tok[0].isalpha():
                L = len(tok)
                out.append(_word_shape(tok) + ("s" if L <= 3 else "m" if L <= 6 else "l")
                           + ("'" if ("'" in tok or "’" in tok) else ""))
            elif tok[0].isdigit():
                out.append("d")
            else:
                out.append(tok)
        docs.append(" ".join(out))
    return docs


# ---------------------------------------------------------------- typography
TYPO_NAMES = ["non_ascii_rate", "curly_share_of_quotes", "curly_share_of_apos", "spaced_em_dash_rate",
              "unspaced_em_dash_rate", "spaced_hyphen_dash_rate", "double_hyphen_rate", "double_space_rate",
              "space_before_punct_rate", "bold_md_rate", "italic_md_rate", "bullet_line_frac", "numbered_line_frac",
              "heading_line_frac", "blank_line_rate", "contraction_rate", "hyphenated_word_rate", "title_case_word_rate",
              "all_caps_word_rate", "oxford_comma_rate", "eg_ie_rate", "percent_rate", "ampersand_rate", "slash_rate",
              "colon_then_capital_rate", "semicolon_per_sentence", "ends_with_period", "ends_with_other",
              "starts_with_quote", "trailing_space_line_frac", "number_token_rate", "ellipsis_char_share"]


def typography_features(texts):
    rows = []
    for t in texts:
        n = max(len(t), 1)
        words = _words(t)
        nw = max(len(words), 1)
        lines = t.split("\n")
        nl = max(len(lines), 1)
        quotes_curly = t.count("“") + t.count("”")
        quotes_all = quotes_curly + t.count('"')
        apos_curly = t.count("’")
        apos_all = apos_curly + t.count("'")
        n_sent = max(len([s for s in re.split(r"[.!?]+\s", t) if s.strip()]), 1)
        ell_char, ell_dots = t.count("…"), t.count("...")
        s = t.rstrip()
        rows.append([
            sum(ord(c) > 127 for c in t) / n,
            quotes_curly / quotes_all if quotes_all else 0.5,
            apos_curly / apos_all if apos_all else 0.5,
            len(re.findall(r" — ", t)) / n, len(re.findall(r"\S—\S", t)) / n, len(re.findall(r" - ", t)) / n,
            t.count("--") / n, t.count("  ") / n, len(re.findall(r" [,.;:!?]", t)) / n,
            len(re.findall(r"\*\*[^*]+\*\*", t)) / n, len(re.findall(r"(?<![*\w])[*_][^*_\n]+[*_](?![*\w])", t)) / n,
            sum(bool(re.match(r"\s*[-*•]\s", l)) for l in lines) / nl,
            sum(bool(re.match(r"\s*\d+[.)]\s", l)) for l in lines) / nl,
            sum(bool(re.match(r"\s*#+\s", l)) for l in lines) / nl,
            t.count("\n\n") / n,
            sum(("'" in w or "’" in w) for w in words) / nw,
            len(re.findall(r"\b[A-Za-z]+-[A-Za-z]+\b", t)) / nw,
            sum(w[0].isupper() and w[1:].islower() for w in words if len(w) > 1) / nw,
            sum(w.isupper() and len(w) > 1 for w in words) / nw,
            len(re.findall(r",\s+(?:and|or)\b", t)) / nw,
            len(re.findall(r"\b(?:e\.g\.|i\.e\.|etc\.)", t)) / nw,
            t.count("%") / n, t.count("&") / n, t.count("/") / n,
            len(re.findall(r":\s+[A-Z]", t)) / n, t.count(";") / n_sent,
            float(s.endswith(".")), float(bool(s) and s[-1] not in ".!?\"”'’)"),
            float(t.lstrip()[:1] in "\"“'‘"),
            sum(l != l.rstrip(" \t") for l in lines) / nl,
            len(re.findall(r"\b\d", t)) / nw,
            ell_char / (ell_char + ell_dots) if (ell_char + ell_dots) else 0.5,
        ])
    return np.asarray(rows, dtype=np.float64)


# ---------------------------------------------------------------- sentence / paragraph structure
_SENT_SPLIT = re.compile(r"(?<=[.!?…])[\"”’)]*\s+(?=[\"“‘(]*[A-Z0-9])")
SENT_NAMES = (["sent_len_mean", "sent_len_std", "sent_len_cv", "sent_len_median", "sent_len_p10", "sent_len_p90",
               "sent_len_min", "sent_len_max", "n_sentences_log", "frac_short_sent", "frac_long_sent",
               "first_sent_len", "last_sent_len", "commas_per_sent", "frac_question_sent", "frac_exclaim_sent",
               "frac_conj_start", "frac_the_start", "frac_pronoun_start", "frac_adverb_ly_start",
               "frac_sent_with_colon", "frac_sent_with_dash", "frac_sent_with_paren",
               "n_paragraphs_log", "sent_per_para_mean", "words_per_para_mean", "words_per_para_std",
               "word_len_std", "long_word_rate", "hapax_rate", "mean_word_freq_rank_proxy"]
              + [f"wordlen_{k}" for k in range(1, 16)])
_CONJ = {"and", "but", "or", "so", "yet", "nor"}
_PRON = {"i", "we", "you", "he", "she", "it", "they", "this", "that", "these", "those"}


def sentence_features(texts):
    rows = []
    for t in texts:
        sents = [s for s in _SENT_SPLIT.split(t.strip()) if s.strip()] or [t]
        sw = [_words(s) for s in sents]
        L = np.array([len(w) for w in sw], dtype=float)
        first = [w[0].lower() if w else "" for w in sw]
        paras = [p for p in re.split(r"\n\s*\n", t.strip()) if p.strip()] or [t]
        pw = np.array([len(_words(p)) for p in paras], dtype=float)
        words = _words(t)
        wl = np.array([len(w) for w in words], dtype=float) if words else np.zeros(1)
        lower = [w.lower() for w in words]
        counts = {}
        for w in lower:
            counts[w] = counts.get(w, 0) + 1
        nw = max(len(words), 1)
        ns = len(sents)
        hist = np.bincount(np.clip(wl.astype(int), 1, 15), minlength=16)[1:16] / nw
        rows.append([
            L.mean(), L.std(), L.std() / max(L.mean(), 1e-9), np.median(L), np.percentile(L, 10), np.percentile(L, 90),
            L.min(), L.max(), np.log1p(ns), (L < 8).mean(), (L > 30).mean(), L[0], L[-1],
            sum(s.count(",") for s in sents) / ns, np.mean([s.rstrip().endswith("?") for s in sents]),
            np.mean([s.rstrip().endswith("!") for s in sents]),
            np.mean([f in _CONJ for f in first]), np.mean([f == "the" for f in first]),
            np.mean([f in _PRON for f in first]), np.mean([f.endswith("ly") and len(f) > 3 for f in first]),
            np.mean([":" in s for s in sents]), np.mean([("—" in s or " - " in s or "–" in s) for s in sents]),
            np.mean(["(" in s for s in sents]),
            np.log1p(len(paras)), ns / len(paras), pw.mean(), pw.std(),
            wl.std(), (wl >= 7).mean(), sum(c == 1 for c in counts.values()) / nw,
            np.mean([w in _FW_INDEX for w in lower]) if lower else 0.0,
            *hist,
        ])
    return np.asarray(rows, dtype=np.float64)


# ---------------------------------------------------------------- group registry
def _dense(fn, clip=None):
    steps = [("f", FunctionTransformer(fn)), ("z", StandardScaler())]
    if clip is not None:
        steps.append(("clip", FunctionTransformer(np.clip, kw_args={"a_min": -clip, "a_max": clip})))
    return Pipeline(steps)


def fw_tokens(text):
    """Function-word tokens of a text (the analyzer of the `fw` group)."""
    return [w for w in (x.lower().replace("’", "'").split("'")[0] for x in _words(text)) if w in _FW_INDEX]


def _tfidf_docs(fn, **kw):
    return Pipeline([("docs", FunctionTransformer(fn)),
                     ("tfidf", TfidfVectorizer(token_pattern=r"\S+", lowercase=False, sublinear_tf=True, **kw))])


def make_group(name):
    if name == "char":
        return TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)
    if name == "word":
        return TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    if name == "style":
        return _dense(style_features)
    if name == "fw":
        # sublinear TF-IDF over the fixed function-word vocabulary; standardised per-word rates gave rare
        # function words huge z-scores on held-out rows (first ablation run: OOF log loss ~12)
        return TfidfVectorizer(analyzer=fw_tokens, vocabulary=FUNCTION_WORDS, sublinear_tf=True)
    if name == "fwng":
        return _tfidf_docs(fw_masked_docs, ngram_range=(1, 3), min_df=3)
    if name == "typo":
        return _dense(typography_features, clip=DENSE_CLIP)
    if name == "sent":
        return _dense(sentence_features, clip=DENSE_CLIP)
    if name == "shape":
        return _tfidf_docs(shape_docs, ngram_range=(1, 3), min_df=3)
    raise ValueError(name)


DENSE_CLIP = 5.0
GROUPS = ["char", "word", "style", "fw", "fwng", "typo", "sent", "shape"]
EXP02_GROUPS = ["char", "word", "style"]
NEW_GROUPS = ["fw", "fwng", "typo", "sent", "shape"]


def make_features(groups):
    """ColumnTransformer over a DataFrame with a 'text' column; block order = `groups` order."""
    return ColumnTransformer([(g, make_group(g), "text") for g in groups])
'''

from scipy.special import log_softmax, softmax

# ---- exp03/stylo/features.py (written as a module: stylo_exp03_model.joblib pickles references to `features`) ----
SRC_DIR = f"{OUT}/h5_src"
os.makedirs(SRC_DIR, exist_ok=True)
with open(f"{SRC_DIR}/features.py", "w") as f:
    f.write(FEATURES_SRC)
sys.path.insert(0, SRC_DIR)
import features  # noqa: F401

# ---- exp03/stylo/protocol.py (inlined: log_proba, logprob_blend) ----
CLASS_NAMES = CONFIG["class_names"]
PROBA_FLOOR = 1e-12


def log_proba(p):
    return np.log(np.clip(np.asarray(p, dtype=np.float64), PROBA_FLOOR, 1.0))


def logprob_blend(log_probs, weights=None):
    """Weighted log-linear pool: softmax(sum_k w_k log p_k). weights=None -> all 1 (equal blend, nothing fitted)."""
    weights = np.ones(len(log_probs)) if weights is None else np.asarray(weights, dtype=np.float64)
    z = sum(w * lp for w, lp in zip(weights, log_probs))
    return np.exp(log_softmax(z, axis=1))

# ---- exp03/probe/probe.py (inlined: ProbeModel) ----
class ProbeModel:
    """Numpy-only probe loaded from the exported .npz."""

    def __init__(self, path):
        with np.load(path, allow_pickle=False) as d:
            self.mean, self.scale = d["mean"], d["scale"]
            self.coef, self.intercept = d["coef"], d["intercept"]
            self.class_names = [str(c) for c in d["class_names"]]
            self.meta = json.loads(str(d["meta"]))

    def decision_function(self, H):
        H = np.asarray(H, dtype=np.float64)
        assert H.ndim == 2 and H.shape[1] == self.mean.shape[0], H.shape
        return ((H - self.mean) / self.scale) @ self.coef.T + self.intercept

    def predict_proba(self, H):
        return softmax(self.decision_function(H), axis=1)

# ---- exp03/qlora_armh/armh_core.py (inlined: pooled_forward) ----
def pooled_forward(model, score_module, tokenizer, texts, batch_size, max_length, n_classes, device="cuda:0"):
    """exp02 pooled_forward: hidden state at the last non-pad token feeding `score`, plus the model logits."""
    captured = {}
    handle = score_module.register_forward_hook(lambda mod, inp, out: captured.__setitem__("h", inp[0].detach()))
    hidden = np.zeros((len(texts), score_module.in_features if hasattr(score_module, "in_features")
                       else score_module.original_module.in_features), dtype=np.float32)
    logits = np.zeros((len(texts), n_classes), dtype=np.float32)
    try:
        with torch.no_grad():
            for start in range(0, len(texts), batch_size):
                batch = texts[start:start + batch_size]
                enc = tokenizer(batch, truncation=True, max_length=max_length, padding=True, return_tensors="pt").to(device)
                with torch.autocast(device.split(":")[0], dtype=torch.float16, enabled=device.startswith("cuda")):
                    out = model(**enc).logits
                ids = enc["input_ids"]
                non_pad = (ids != tokenizer.pad_token_id).int()
                last = (torch.arange(ids.shape[-1], device=ids.device) * non_pad).argmax(-1)
                h = captured["h"][torch.arange(ids.shape[0], device=ids.device), last]
                hidden[start:start + len(batch)] = h.float().cpu().numpy()
                logits[start:start + len(batch)] = out.float().cpu().numpy()
    finally:
        handle.remove()
    return hidden, logits

# ---- submission/h5/h5_core.py (inlined) ----
"""H5 submission logic that runs after the shared Qwen forward pass (CPU; inlined into the Kaggle notebook by
build_notebook.py, tested offline in tests/).

H5 = equal-weight log-linear pool of three components, exactly as in exp03/qlora_armh/analyze_armh.py:
  probe_armh   C=0.03 StandardScaler+LR probe (probe_armh.npz) on the pooled score-input hidden state
  armh_head    softmax of the epoch-6 Arm H classification head logits
  stylo_exp03  char+word+style TF-IDF/LR pipeline (stylo_exp03_model.joblib)
blend = protocol.logprob_blend([log p_probe, log p_head, log p_stylo]) (weights all 1, nothing fitted).
"""
import numpy as np
import pandas as pd
from scipy.special import softmax
from sklearn.metrics import balanced_accuracy_score

H5_COMPONENTS = ("probe_armh", "armh_head", "stylo_exp03")


def component_probas(hidden, logits, texts, probe, stylo):
    """Per-component probabilities (n, 18) in CLASS_NAMES order from one shared forward pass.

    The probe was fit and scored on pooled_hidden_armh.npy, which the Arm H kernel stored as float16, so the
    hidden states take the same float32 -> float16 -> float64 path here."""
    H = np.asarray(hidden, dtype=np.float32).astype(np.float16).astype(np.float64)
    return {"probe_armh": probe.predict_proba(H),
            "armh_head": softmax(np.asarray(logits, dtype=np.float32).astype(np.float64), axis=1),
            "stylo_exp03": stylo.predict_proba(pd.DataFrame({"text": list(texts)}))}


def h5_blend(comps):
    """Fixed equal-weight H5 blend (not the H6 dev-LOPO weights)."""
    return logprob_blend([log_proba(comps[k]) for k in H5_COMPONENTS])


def check_class_order(label_map_names, model_config_names, probe_names, stylo_classes, train_models=None):
    """Every component must use the alphabetical 18-class order (class id c = c-th sorted model name).

    train_models (train.csv "model") is only checked when given, i.e. when the optional dev check reads train.csv."""
    names = list(CLASS_NAMES)
    assert len(names) == 18 and names == sorted(names)
    assert list(label_map_names) == names, "adapter label_map.json order differs"
    assert list(model_config_names) == names, "model.config.id2label order differs"
    assert list(probe_names) == names, "probe_armh.npz class_names order differs"
    assert [int(c) for c in stylo_classes] == list(range(18)), "stylo classes_ are not label ids 0..17"
    if train_models is not None:
        assert sorted(set(train_models)) == names, "train.csv model names differ from CLASS_NAMES"
    return names


def dev_reproduction(y, blend, comps, ref_h5, ref_shared, ref_comps, expected_bal_acc, tol, max_dp_shared):
    """Compare the dev run against blend_H5_dev.csv (the reported H5) and the shared-pass reference.

    Returns (report, failures). failures is empty when the run reproduces H5."""
    pred = blend.argmax(1)
    bal = float(balanced_accuracy_score(y, pred))
    rep = {"balanced_accuracy": bal, "expected_balanced_accuracy": expected_bal_acc, "tolerance": tol,
           "argmax_agreement_vs_blend_H5_dev": float((pred == ref_h5.argmax(1)).mean()),
           "rows_differing_argmax_vs_blend_H5_dev": np.where(pred != ref_h5.argmax(1))[0].tolist(),
           "max_abs_proba_diff_vs_blend_H5_dev": float(np.abs(blend - ref_h5).max()),
           "max_abs_proba_diff_vs_shared_pass_reference": float(np.abs(blend - ref_shared).max()),
           "components": {k: {"balanced_accuracy": float(balanced_accuracy_score(y, comps[k].argmax(1))),
                              "max_abs_proba_diff_vs_reference": float(np.abs(comps[k] - ref_comps[k]).max()),
                              "argmax_agreement_vs_reference": float((comps[k].argmax(1) == ref_comps[k].argmax(1)).mean())}
                          for k in H5_COMPONENTS}}
    fails = []
    if abs(bal - expected_bal_acc) > tol:
        fails.append(f"dev balanced accuracy {bal:.6f} differs from {expected_bal_acc:.6f} by more than {tol}")
    if rep["argmax_agreement_vs_blend_H5_dev"] < 1.0:
        fails.append(f"argmax differs from blend_H5_dev.csv on rows {rep['rows_differing_argmax_vs_blend_H5_dev']}")
    if rep["max_abs_proba_diff_vs_shared_pass_reference"] > max_dp_shared:
        fails.append(f"max |p - shared-pass reference| {rep['max_abs_proba_diff_vs_shared_pass_reference']:.3g} > {max_dp_shared}")
    return rep, fails


def validate_submission(sub, test, class_names):
    assert list(sub.columns) == ["id", "model"], list(sub.columns)
    assert len(sub) == len(test), (len(sub), len(test))
    assert sub["id"].notna().all() and sub["id"].is_unique, "ids must be unique and non-null"
    assert sub["id"].tolist() == test["id"].tolist(), "ids must match test.csv (same order)"
    assert sub["model"].notna().all() and set(sub["model"]) <= set(class_names), "labels outside the 18 classes"
    return True


def extrapolate(batch_seconds, batch_rows, n_rows=4000, batch_size=8):
    """Linear extrapolation of the shared forward pass from measured per-batch wall-clock."""
    secs, rows = np.asarray(batch_seconds, dtype=float), np.asarray(batch_rows, dtype=float)
    per_row = secs.sum() / rows.sum()
    return {"measured_batches": int(len(secs)), "measured_rows": int(rows.sum()), "measured_seconds": float(secs.sum()),
            "mean_seconds_per_batch": float(secs.mean()), "p95_seconds_per_batch": float(np.percentile(secs, 95)),
            "seconds_per_row": float(per_row), "rows": n_rows, "batches": int(np.ceil(n_rows / batch_size)),
            "estimated_forward_minutes": float(per_row * n_rows / 60)}


In [ ]:
import joblib
from sklearn.model_selection import GroupShuffleSplit

checks = {"adapter/adapter_model.safetensors": f"{ADAPTER_DIR}/adapter_model.safetensors",
          "adapter/adapter_config.json": f"{ADAPTER_DIR}/adapter_config.json",
          "adapter/label_map.json": f"{ADAPTER_DIR}/label_map.json",
          **{k: f"{H5_DIR}/{k}" for k in ["probe_armh.npz", "stylo_exp03_model.joblib", "blend_H5_dev.csv", "h5_dev_reference.csv"]}}
for k, path in checks.items():
    got = sha256_file(path)
    assert got == SHA[k], f"{k}: sha256 {got} != expected {SHA[k]}"
print("all artifact sha256 verified:", list(checks))

test = pd.read_csv(f"{DATA_DIR}/test.csv")  # row count not assumed: the hidden rerun has ~4,000 rows
assert {"id", "text"} <= set(test.columns) and len(test) > 0
print("test:", test.shape)

label2id = {n: i for i, n in enumerate(CLASS_NAMES)}
id2label = {i: n for n, i in label2id.items()}
if RUN_DEV_CHECK:
    TRAIN_CSV = f"{DATA_DIR}/train.csv"
    if not os.path.exists(TRAIN_CSV):
        print(f"WARNING: run_dev_check=true but {TRAIN_CSV} does not exist; dev check skipped")
        RUN_DEV_CHECK = False
if RUN_DEV_CHECK:
    got = sha256_file(TRAIN_CSV)
    assert got == SHA["train.csv"], f"train.csv: sha256 {got} != expected {SHA['train.csv']}"
    train = pd.read_csv(TRAIN_CSV)
    df = train[["id", "prompt_id", "text", "model"]].copy()
    df["labels"] = df["model"].map(label2id).astype(int)
    _, va_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED).split(df, groups=df["prompt_id"]))
    dev_df = df.iloc[va_idx].reset_index(drop=True)
    assert (len(dev_df), dev_df["prompt_id"].nunique()) == (DC["rows"], DC["prompts"])
    assert (dev_df["model"].value_counts() == 23).all() and dev_df["model"].nunique() == N_CLASSES
    print("train.csv sha256 verified; dev split verified: 414 rows / 23 prompts (exp01 split)")
print("run_dev_check:", RUN_DEV_CHECK)

with open(f"{ADAPTER_DIR}/label_map.json") as f:
    label_map_names = [n for _, n in sorted((int(k), v) for k, v in json.load(f)["id2label"].items())]
with open(f"{ADAPTER_DIR}/adapter_config.json") as f:
    adapter_cfg = json.load(f)
# PEFT appends its SEQ_CLS defaults ("classifier", "score") to the trained modules_to_save=["score"].
assert "score" in adapter_cfg["modules_to_save"], adapter_cfg["modules_to_save"]
assert set(adapter_cfg["modules_to_save"]) <= {"score", "classifier"}, adapter_cfg["modules_to_save"]
assert adapter_cfg["task_type"] == "SEQ_CLS", adapter_cfg["task_type"]
probe = ProbeModel(f"{H5_DIR}/probe_armh.npz")
assert probe.meta["C"] == 0.03 and probe.meta["model"] == "probe_armh", probe.meta.get("C")
stylo = joblib.load(f"{H5_DIR}/stylo_exp03_model.joblib")
print("probe C:", probe.meta["C"], "| coef", probe.coef.shape, "| stylo:", [s for s, _ in stylo.steps])


In [ ]:
from peft import PeftModel, prepare_model_for_kbit_training
from transformers import AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig, set_seed

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
                          bnb_4bit_use_double_quant=True)
set_seed(SEED)
base = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH, num_labels=N_CLASSES, id2label=id2label, label2id=label2id, quantization_config=qcfg,
    device_map={"": 0}, torch_dtype=torch.float16, local_files_only=True)
base.config.pad_token_id = tokenizer.pad_token_id
assert type(base).__name__ == "Qwen3ForSequenceClassification", type(base).__name__
# Same dtype state as the Arm H kernel that produced the probe features and head logits: prepare_model_for_kbit_training
# upcasts the non-quantized params (embeddings, RMSNorms, score) to fp32. Without it the dev check fails
# (kernel v2: bal-acc 0.4300, max |dp| 0.017). No gradient checkpointing, all params frozen.
base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=False)
assert base.model.norm.weight.dtype == torch.float32 and base.score.weight.dtype == torch.float32
model = PeftModel.from_pretrained(base, ADAPTER_DIR, is_trainable=False)
assert not any(p.requires_grad for p in model.parameters())
model.eval()
score_wrapper = model.base_model.model.score  # ModulesToSaveWrapper; the hook captures its input
assert "default" in score_wrapper.modules_to_save
load_minutes = (time.time() - t0) / 60
print(f"model + adapter loaded in {load_minutes:.1f} min | GPU allocated {torch.cuda.memory_allocated() / 1e9:.2f} GB")

model_config_names = [model.config.id2label[i] for i in range(N_CLASSES)]
check_class_order(label_map_names, model_config_names, probe.class_names, stylo.classes_)
if RUN_DEV_CHECK:
    check_class_order(label_map_names, model_config_names, probe.class_names, stylo.classes_, train["model"])
print("class order identical across CLASS_NAMES, head config, label_map.json, probe npz and stylo classes_:", CLASS_NAMES)


In [ ]:
batch_log = []


def shared_forward(texts, split):
    """One forward pass per batch of 8 (file order): pooled_forward returns head logits + pooled hidden states."""
    hidden = np.zeros((len(texts), probe.mean.shape[0]), dtype=np.float32)
    logits = np.zeros((len(texts), N_CLASSES), dtype=np.float32)
    for start in range(0, len(texts), BATCH):
        chunk = texts[start:start + BATCH]
        t = time.perf_counter()
        h, lg = pooled_forward(model, score_wrapper, tokenizer, chunk, BATCH, MAX_LENGTH, N_CLASSES)
        torch.cuda.synchronize()
        dt = time.perf_counter() - t
        hidden[start:start + len(chunk)], logits[start:start + len(chunk)] = h, lg
        batch_log.append({"split": split, "start": start, "rows": len(chunk), "seconds": dt,
                          "peak_allocated_gb": torch.cuda.max_memory_allocated() / 1e9})
        if (start // BATCH) % 25 == 0:
            print(f"[{split}] batch {start // BATCH + 1}/{-(-len(texts) // BATCH)}: {dt:.2f}s")
    return hidden, logits


def predict_h5(texts, split):
    hidden, logits = shared_forward(list(texts), split)
    t = time.perf_counter()
    comps = component_probas(hidden, logits, texts, probe, stylo)
    blend = h5_blend(comps)
    print(f"[{split}] probe + stylo + blend on CPU: {time.perf_counter() - t:.1f}s")
    return comps, blend, logits


torch.cuda.reset_peak_memory_stats()


In [ ]:
# Dev reproduction (414 rows, only when run_dev_check=true): same inference path, compared with the reported H5
# before any test prediction. Saved version 3 passed it end-to-end (bal-acc 0.432367, 414/414 argmax, max |dp| 5e-11).
if RUN_DEV_CHECK:
    t0 = time.time()
    dev_comps, dev_blend, dev_logits = predict_h5(dev_df["text"].tolist(), "dev")
    dev_minutes = (time.time() - t0) / 60

    def aligned(path):
        r = pd.read_csv(path).set_index("id").loc[dev_df["id"]].reset_index()
        assert (r["labels"].to_numpy() == dev_df["labels"].to_numpy()).all()
        return r

    PCOLS = [f"p_{n}" for n in CLASS_NAMES]
    h5_rep = aligned(f"{H5_DIR}/blend_H5_dev.csv")[PCOLS].to_numpy(np.float64)
    h5_rep = h5_rep / h5_rep.sum(1, keepdims=True)
    ref = aligned(f"{H5_DIR}/h5_dev_reference.csv")
    ref_shared = ref[[f"h5:{c}" for c in PCOLS]].to_numpy(np.float64)
    ref_comps = {k: ref[[f"{k}:{c}" for c in PCOLS]].to_numpy(np.float64) for k in H5_COMPONENTS}
    dev_report, dev_failures = dev_reproduction(
        dev_df["labels"].to_numpy(), dev_blend, dev_comps, h5_rep, ref_shared, ref_comps,
        DC["expected_balanced_accuracy"], DC["tolerance"], DC["max_abs_proba_diff_vs_shared_pass_reference"])
    dev_report["minutes"] = dev_minutes
    print(json.dumps(dev_report, indent=1))
    out = dev_df[["id", "prompt_id", "labels"]].copy()
    out = pd.concat([out, pd.DataFrame(dev_blend, columns=PCOLS)], axis=1)
    out.to_csv(f"{OUT}/h5_dev_predictions.csv", index=False, float_format="%.10g")
    with open(f"{OUT}/h5_dev_check.json", "w") as f:
        json.dump(dev_report, f, indent=1)
    if dev_failures:
        raise RuntimeError("dev reproduction failed; investigate before writing a submission: " + "; ".join(dev_failures))
    print(f"dev reproduction OK: balanced accuracy {dev_report['balanced_accuracy']:.4f} "
          f"(expected {DC['expected_balanced_accuracy']:.4f} +/- {DC['tolerance']}), argmax identical to blend_H5_dev.csv")
else:
    dev_report = {"skipped": True}
    dev_minutes = 0
    print("dev check skipped (run_dev_check=false)")


In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

text_ok = test["text"].map(lambda v: isinstance(v, str)).astype(bool)
print(f"test.csv: {int((~text_ok).sum())} null/non-string text rows coerced to empty string")
test["text"] = test["text"].where(text_ok, "").fillna("")

t0 = time.time()
test_comps, test_blend, _ = predict_h5(test["text"].tolist(), "test")
test_minutes = (time.time() - t0) / 60
submission = pd.DataFrame({"id": test["id"].to_numpy(), "model": [CLASS_NAMES[i] for i in test_blend.argmax(1)]})
validate_submission(submission, test, CLASS_NAMES)
sample_schema = pq.read_schema(f"{DATA_DIR}/sample_submission.parquet")
assert sample_schema.names == ["id", "model"], sample_schema
pq.write_table(pa.Table.from_pandas(submission, preserve_index=False).cast(sample_schema), f"{OUT}/submission.parquet")

written = pd.read_parquet(f"{OUT}/submission.parquet")
validate_submission(written, test, CLASS_NAMES)
assert pq.read_schema(f"{OUT}/submission.parquet").equals(sample_schema, check_metadata=False)
print(f"submission.parquet written and validated: {len(written)} rows, columns {list(written.columns)}, {test_minutes:.2f} min")

# Diagnostics only from here on: submission.parquet is final, so nothing below may raise.
try:
    print(written.head(10))
    print(written["model"].value_counts().to_string())
except Exception as e:
    print(f"WARNING: post-submission diagnostics failed: {type(e).__name__}: {e}")


In [ ]:
# Post-submission diagnostics (batch log, runtime extrapolation, run summary): warnings only, never raises.
import urllib.request

try:
    try:
        urllib.request.urlopen("https://pypi.org", timeout=5)
        internet = "REACHABLE"
    except Exception as e:
        internet = f"unreachable ({type(e).__name__})"
    print("internet:", internet)

    bl = pd.DataFrame(batch_log)
    bl.to_csv(f"{OUT}/h5_batch_log.csv", index=False)
    peak = {"allocated_gb": torch.cuda.max_memory_allocated() / 1e9, "reserved_gb": torch.cuda.max_memory_reserved() / 1e9}
    est = extrapolate(bl["seconds"], bl["rows"], n_rows=4000, batch_size=BATCH)
    total_minutes = (time.time() - RUN_START) / 60
    est["estimated_total_minutes_4000_rows"] = (total_minutes - test_minutes) + est["estimated_forward_minutes"]
    summary = {"status": "completed", "test_rows": len(test), "run_dev_check": RUN_DEV_CHECK,
               "dev_balanced_accuracy": dev_report.get("balanced_accuracy"),
               "dev_check": dev_report, "peak_gpu_memory": peak, "model_load_minutes": load_minutes,
               "dev_minutes": dev_minutes, "test_minutes": test_minutes, "total_minutes": total_minutes,
               "per_batch_seconds": {s: g["seconds"].describe().to_dict() for s, g in bl.groupby("split")},
               "extrapolation": est, "internet": internet, "versions": versions, "config": CONFIG}
    with open(f"{OUT}/h5_run_summary.json", "w") as f:
        json.dump(summary, f, indent=1, default=float)
    print(json.dumps({k: v for k, v in summary.items() if k not in ("config", "dev_check")}, indent=1, default=float))
except Exception as e:
    print(f"WARNING: run summary failed (submission.parquet already written): {type(e).__name__}: {e}")
